# m1 — Data Acquisition

Downloads sequencing reads for all samples in the batch manifest.

| Step | Tool | Detail |
|------|------|--------|
| Config load | JSON / papermill params | Resolve `SAMPLE_CSV`, `INPUT_SOURCE` |
| Parallel download | `fasterq-dump` + `ThreadPoolExecutor` | Up to 4 concurrent SRA downloads |
| Manifest write | pandas | Save per-sample download status to Drive |

**Checkpointing**: samples whose FASTQ files already exist on disk are skipped.

In [ ]:
# parameters
SAMPLE_CSV      = ""          # path to CSV with columns: srr, sample_label, phenotype
SRR_LIST        = []          # alternative: explicit list of SRR accessions
SRR_ACCESSION   = ""          # alternative: single SRR accession (v2 compat)
DRIVE_OUTPUT    = "mmpR5_pipeline/output"
DRIVE_REF       = "mmpR5_pipeline/input"
INPUT_SOURCE    = "sra"       # "sra" | "upload"
READ_TYPE       = "illumina"  # "illumina" | "nanopore" | "pacbio"

In [ ]:
# CPU only — no GPU needed
# ── Load pipeline config (Drive JSON fallback) ──────────────────────────────
import json, shutil, subprocess, warnings, time, datetime, concurrent.futures
from pathlib import Path
from Bio import SeqIO, Entrez
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
import pandas as pd
import numpy as np
import requests
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

_CFG_PATH = Path("/content/drive/MyDrive/ColabNotebooks/mmpR5_pipeline/pipeline_config.json")

def _load_config():
    if _CFG_PATH.exists():
        with open(_CFG_PATH) as _f:
            return json.load(_f)
    return {}

_cfg = _load_config()

def _p(key, default=None):
    """Resolve parameter: papermill-injected variable takes precedence over config JSON."""
    try:
        v = eval(key)                # injected by papermill
        return v if v is not None else _cfg.get(key, default)
    except Exception:
        return _cfg.get(key, default)

In [ ]:
# CPU only — no GPU needed
from google.colab import drive
drive.mount("/content/drive")
DRIVE_BASE  = Path("/content/drive/MyDrive/ColabNotebooks")
OUTPUT_ROOT = DRIVE_BASE / _p("DRIVE_OUTPUT", "mmpR5_pipeline/output")
REF_DIR     = DRIVE_BASE / _p("DRIVE_REF",    "mmpR5_pipeline/input")
MODULES_DIR = DRIVE_BASE / "mmpR5_pipeline" / "modules"
print(f"Drive mounted. Output root: {OUTPUT_ROOT}")

In [ ]:
# CPU only — no GPU needed
# ── Resolve sample manifest ───────────────────────────────────────────────────
_sample_csv   = _p("SAMPLE_CSV", "")
_srr_list     = _p("SRR_LIST", [])
_srr_single   = _p("SRR_ACCESSION", "")
_input_source = _p("INPUT_SOURCE", "sra")

_sample_manifest = []
if _sample_csv:
    _csv_df = pd.read_csv(_sample_csv)
    for _, _r in _csv_df.iterrows():
        _srr  = str(_r["srr"]).strip()
        _lbl  = str(_r.get("sample_label", _srr)).strip()
        _phen = str(_r.get("phenotype", "U")).strip().upper()
        _sample_manifest.append({"srr": _srr, "label": _lbl or _srr, "phenotype": _phen})
elif _srr_list:
    for _s in _srr_list:
        _sample_manifest.append({"srr": _s.strip(), "label": _s.strip(), "phenotype": "U"})
elif _srr_single:
    _sample_manifest.append({"srr": _srr_single, "label": _srr_single, "phenotype": "U"})
else:
    raise ValueError("No samples configured. Set SAMPLE_CSV, SRR_LIST, or SRR_ACCESSION.")

# Create per-sample work dirs and output subdirs
for _sm in _sample_manifest:
    Path(f"/content/{_sm['label']}").mkdir(parents=True, exist_ok=True)
    for _s in ["01_qc","02_alignment","03_variants","04_sequences","05_structure","06_scores"]:
        (OUTPUT_ROOT / _sm["label"] / _s).mkdir(parents=True, exist_ok=True)
(OUTPUT_ROOT / "07_ml_results").mkdir(parents=True, exist_ok=True)

print(f"Sample manifest: {len(_sample_manifest)} sample(s)")
for _sm in _sample_manifest:
    print(f"  {_sm['label']}  ({_sm['srr']})  phenotype={_sm['phenotype']}")

In [ ]:
# CPU only — no GPU needed
# ── Parallel SRA download ──────────────────────────────────────────────────────
def _download_one(sm):
    srr, label = sm["srr"], sm["label"]
    wdir = Path(f"/content/{label}")
    r1p  = wdir / f"{srr}_1.fastq"
    rsg  = wdir / f"{srr}.fastq"
    if r1p.exists() or rsg.exists():
        return (label, True, "already on disk")
    try:
        res = subprocess.run(
            ["fasterq-dump", "--split-files", srr, "--outdir", str(wdir)],
            capture_output=True, text=True, timeout=3600)
        return (label, res.returncode == 0,
                "OK" if res.returncode == 0 else res.stderr[:300])
    except Exception as exc:
        return (label, False, str(exc))

if _input_source == "sra":
    _nw = min(4, len(_sample_manifest))
    print(f"Parallel download: {len(_sample_manifest)} sample(s)  (workers={_nw})")
    _dl_status = {}
    with concurrent.futures.ThreadPoolExecutor(max_workers=_nw) as _pool:
        _futs = {_pool.submit(_download_one, sm): sm for sm in _sample_manifest}
        for _fut in concurrent.futures.as_completed(_futs):
            _lbl, _ok, _msg = _fut.result()
            _dl_status[_lbl] = "OK" if _ok else "FAILED"
            print(f"  [{'OK  ' if _ok else 'FAIL'}] {_lbl}: {_msg}")
else:
    print("INPUT_SOURCE='upload' — skipping batch download.")
    _dl_status = {sm["label"]: "upload" for sm in _sample_manifest}

# Save manifest + download status to Drive
_manifest_df = pd.DataFrame([
    {"srr": sm["srr"], "sample_label": sm["label"],
     "phenotype": sm["phenotype"], "download_status": _dl_status.get(sm["label"], "?")}
    for sm in _sample_manifest
])
_manifest_csv = OUTPUT_ROOT / "sample_manifest.csv"
_manifest_df.to_csv(str(_manifest_csv), index=False)
print(f"\nManifest saved: {_manifest_csv}")
display(_manifest_df)